In [ ]:
# =====================================================================
# Dynamic-Room LGBM Training v2
#   Features: 17 base + sinc phase + Lx/λ, Lz/λ = 21 features
#   Data: 30k LHS + 300 Pareto frontier + 2000 boundary oversample
#   Params: linear_tree=True, extra_trees=True, min_data_in_leaf=40
# =====================================================================

import numpy as np, lightgbm as lgb, optuna, json, time, os, glob, math
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats.qmc import LatinHypercube
import warnings; warnings.filterwarnings('ignore')

# ==========================================
# 1. Load base data (30k LHS, 17 features)
# ==========================================
paths = ['dynamic_train_enriched.csv','/kaggle/working/dynamic_train_enriched.csv'] + sorted(glob.glob('/kaggle/input/**/dynamic_train_enriched.csv', recursive=True))
fpath = None
for p in paths:
    if os.path.exists(p): fpath = p; break
if fpath is None: raise FileNotFoundError('dynamic_train_enriched.csv')
print(f'Loading {fpath}...')
data = np.loadtxt(fpath, delimiter=',', skiprows=1)
X_base = data[:, :-1]; Y_base = data[:, -1]
print(f'Base: {X_base.shape[0]} samples, {X_base.shape[1]} features, feasible={(Y_base<0.10).sum()/len(Y_base)*100:.1f}%')

# ==========================================
# 2. Add sinc phase features (from existing columns)
#    CSV cols: xc,zc,Lx,Lz,rx,ry,xc_rel,Lx_rel,area_ratio,dist_bs_win,alpha_x,alpha_z,...
Lx = X_base[:,2]; Lz = X_base[:,3]; alpha_x = X_base[:,10]; alpha_z = X_base[:,11]
lam_val = 0.075
phase_x = (math.pi/lam_val) * Lx * alpha_x   # sinc argument for horizontal diffraction
phase_z = (math.pi/lam_val) * Lz * alpha_z   # sinc argument for vertical diffraction
Lx_over_lam = Lx / lam_val
Lz_over_lam = Lz / lam_val
X = np.column_stack([X_base, phase_x, phase_z, Lx_over_lam, Lz_over_lam])
Y = Y_base
print(f'After phase features: {X.shape[1]} features (added: phase_x, phase_z, Lx/λ, Lz/λ)')

# ==========================================
# 3. Bound-zone feature: near 10% cliff (8-15%)
# ==========================================
near_cliff = (Y >= 0.08) & (Y <= 0.15)
print(f'Near 10% cliff (8-15%): {near_cliff.sum()} samples ({near_cliff.sum()/len(Y)*100:.1f}%)')

# ==========================================
# 4. Pareto Frontier Augmentation (300 pts)
#    Use quick NSGA-II to generate boundary samples
# ==========================================
print('\n--- Pareto Frontier Augmentation ---')
# We generate using physics engine if available, else sample proxy
# Since physics engine requires GPU+room build, use smart proxy:
# Take existing samples near feasible boundary and perturb
feas_mask = Y < 0.12
if feas_mask.sum() > 500:
    # Use existing feasible samples as proxy frontier
    X_feas = X[feas_mask]; Y_feas = Y[feas_mask]
    # Sort by area (Lx*Lz) and take best
    areas = X_feas[:,2]*X_feas[:,3]
    # Spread: take 300 spread across outage range
    idx = np.argsort(Y_feas)
    step = max(1, len(idx)//300)
    pareto_idx = idx[::step][:300]
    X_pareto = X_feas[pareto_idx]; Y_pareto = Y_feas[pareto_idx]
    print(f'Pareto proxy: {len(X_pareto)} pts from feasible samples')
else:
    X_pareto = np.zeros((0, X.shape[1])); Y_pareto = np.zeros(0)
    print('Pareto proxy: insufficient feasible samples, skipping')

# ==========================================
# 5. Boundary Oversample (2k near 8-12%)
# ==========================================
cliff_mask = (Y >= 0.08) & (Y <= 0.12)
if cliff_mask.sum() > 2000:
    idx_c = np.where(cliff_mask)[0]
    np.random.seed(99)
    idx_c = np.random.choice(idx_c, min(2000, len(idx_c)), replace=False)
    X_bound = X[idx_c]; Y_bound = Y[idx_c]
    print(f'Boundary oversample: {len(X_bound)} pts (8-12% outage)')
else:
    X_bound = np.zeros((0, X.shape[1])); Y_bound = np.zeros(0)
    print(f'Boundary: only {cliff_mask.sum()} pts in 8-12%, skipping')

# ==========================================
# 6. Combine training set
# ==========================================
X_train = np.vstack([X, X_pareto, X_bound])
Y_train = np.concatenate([Y, Y_pareto, Y_bound])
print(f'\nFull training: {len(X_train)} samples, feasible={(Y_train<0.10).sum()/len(Y_train)*100:.1f}%')

# ==========================================
# 7. Train/Test Split
# ==========================================
np.random.seed(42); idx=np.random.permutation(len(X_train))
sp=int(len(X_train)*0.85)
Xtr,Xte=X_train[idx[:sp]],X_train[idx[sp:]]
Ytr,Yte=Y_train[idx[:sp]],Y_train[idx[sp:]]

# ==========================================
# 8. Optuna Tuning
# ==========================================
kf=KFold(n_splits=5,shuffle=True,random_state=42)
def objective(trial):
    params={
        'objective':'regression','metric':'mae','verbosity':-1,'device':'gpu',
        'linear_tree':True,'extra_trees':True,
        'learning_rate':trial.suggest_float('learning_rate',0.01,0.1,log=True),
        'max_depth':trial.suggest_int('max_depth',8,16),
        'num_leaves':trial.suggest_int('num_leaves',64,256),
        'min_data_in_leaf':trial.suggest_int('min_data_in_leaf',20,80),
        'feature_fraction':trial.suggest_float('feature_fraction',0.6,0.95),
        'bagging_fraction':trial.suggest_float('bagging_fraction',0.6,0.9),
        'bagging_freq':1,'n_jobs':-1,
    }
    maes=[]
    for ti,vi in kf.split(Xtr):
        Xt,Xv=Xtr[ti],Xtr[vi];Yt,Yv=Ytr[ti],Ytr[vi]
        m=lgb.LGBMRegressor(**params,n_estimators=500,early_stopping_round=30)
        m.fit(Xt,Yt,eval_set=[(Xv,Yv)])
        p=m.predict(Xv);feas=Yv<0.15
        maes.append(mean_absolute_error(Yv[feas],p[feas]) if feas.sum()>0 else mean_absolute_error(Yv,p))
    return np.mean(maes)

print('\nOptuna search (40 trials, 5-Fold CV, linear_tree+extra_trees)...')
study=optuna.create_study(direction='minimize')
study.optimize(objective,n_trials=40,show_progress_bar=True)
print(f'\nBest params: {study.best_params}')
print(f'Best feasible MAE: {study.best_value*100:.2f}%')

# ==========================================
# 9. Final Model
# ==========================================
best=study.best_params
for k in ['objective','metric','verbosity','device','linear_tree','extra_trees','bagging_freq']:
    best[k]=['regression','mae',-1,'gpu',True,True,1][['objective','metric','verbosity','device','linear_tree','extra_trees','bagging_freq'].index(k)]
best['n_jobs']=-1
print('\nTraining final model...')
model=lgb.LGBMRegressor(**best,n_estimators=1000,early_stopping_round=50)
model.fit(Xtr,Ytr,eval_set=[(Xte,Yte)])
pred=model.predict(Xte)
r2_all=r2_score(Yte,pred);mae_all=mean_absolute_error(Yte,pred)
feas=Yte<0.10;r2_feas=r2_score(Yte[feas],pred[feas]) if feas.sum()>1 else 0
mae_feas=mean_absolute_error(Yte[feas],pred[feas]) if feas.sum()>0 else 0
print(f'\nFinal LGBM metrics:')
print(f'  R² all:      {r2_all:.4f}')
print(f'  MAE all:     {mae_all*100:.2f}%')
print(f'  R² feasible: {r2_feas:.4f}')
print(f'  MAE feasible:{mae_feas*100:.2f}%')

# ==========================================
# 10. Save
# ==========================================
model.booster_.save_model('lgbm_dynamic_v2.txt')
meta={'n_features':X.shape[1],'r2_all':r2_all,'mae_feas':mae_feas,'best_params':best}
with open('lgbm_dynamic_v2_meta.json','w') as f:json.dump(meta,f,indent=2)
print('\nModel saved: lgbm_dynamic_v2.txt + lgbm_dynamic_v2_meta.json')
from IPython.display import FileLink,display
display(FileLink('lgbm_dynamic_v2.txt'));display(FileLink('lgbm_dynamic_v2_meta.json'))